# # 03 - Model 2: Reimbursement / Expected-Recovery Regression
#
# **Objective:** predict the expected monetary outcome of a denied claim.
#
# ### Important limitation (documented, not fabricated)
# The DenialIQ dataset has **no paid / reimbursement amount** column for `paid` or
# `partial_pay` claims - only the *billed* `claim_amount_usd`. A true reimbursement
# target therefore **does not exist**. The only real monetary outcome label available
# is **`estimated_recovery_usd`** (expected $ recovered if a denied claim is appealed),
# which is provided for all 33,664 denied claims in `denial_labels.csv`.
#
# We therefore train an **Expected-Recovery regressor on `estimated_recovery_usd`**
# (denied claims only) and clearly label it as such in the API and UI. No target is
# invented. For claims that are *not* denied there is no recovery, so the model is only
# applied to claims the denial model flags as at risk.
#
# * Algorithm: tuned **XGBoost regressor** inside a Scikit-learn pipeline.
# * Features: only the **5 features selected** in notebook 01.
# * Tuning: **RandomizedSearchCV (n_iter=10)**, 3-fold CV.


# ## 1. Imports & paths


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42
BASE = Path.cwd()
while not (BASE / "data" / "processed").exists() and BASE != BASE.parent:
    BASE = BASE.parent
PROCESSED = BASE / "data" / "processed"
MODELS = BASE / "models"
REPORTS = BASE / "reports"
MODELS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)


# ## 2. Load & filter to denied claims (the only rows with a recovery label)


In [2]:
df = pd.read_csv(PROCESSED / "claims_model_ready.csv")
with open(PROCESSED / "feature_columns.json", encoding="utf-8") as fh:
    cols = json.load(fh)
# Only the 5 features selected by the feature-selection step in notebook 01.
CATEGORICAL, NUMERIC, FEATURES = (cols["selected_categorical"],
                                  cols["selected_numeric"], cols["selected_features"])
print("Selected features (%d):" % len(FEATURES), FEATURES)

data = df[(df["outcome"] == "denied") & (df["estimated_recovery_usd"].notna())].copy()
print("Denied claims with recovery label:", data.shape)
print(data["estimated_recovery_usd"].describe())
print("Share of zero-recovery claims:", round((data["estimated_recovery_usd"] == 0).mean(), 3))


Selected features (5): ['documentation_completeness', 'cpt_code', 'primary_icd10_dx', 'claim_amount_usd', 'auth_status']
Denied claims with recovery label: (33664, 40)
count    33664.000000
mean       965.580381
std       2125.715030
min          0.000000
25%         78.390000
50%        178.065000
75%        946.085000
max      30265.170000
Name: estimated_recovery_usd, dtype: float64
Share of zero-recovery claims: 0.147


# ## 3. Outlier / distribution analysis of the target


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data["estimated_recovery_usd"], bins=60, ax=axes[0])
axes[0].set_title("estimated_recovery_usd (raw)")
sns.histplot(np.log1p(data["estimated_recovery_usd"]), bins=60, ax=axes[1])
axes[1].set_title("log1p(estimated_recovery_usd)")
plt.tight_layout(); plt.savefig(REPORTS / "03_target_distribution.png", dpi=110); plt.show()


C:\Windows\Temp\ipykernel_31500\2832726083.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "03_target_distribution.png", dpi=110); plt.show()


# ## 4. Split & preprocessing


In [4]:
X = data[FEATURES]
y = data["estimated_recovery_usd"]

X_train, y_train = X[data["split"] == "train"], y[data["split"] == "train"]
X_val, y_val = X[data["split"] == "val"], y[data["split"] == "val"]
X_test, y_test = X[data["split"] == "test"], y[data["split"] == "test"]
print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_pipe, NUMERIC),
                                  ("cat", categorical_pipe, CATEGORICAL)])


train/val/test: (23597, 5) (4965, 5) (5102, 5)


# ## 5. Hyper-parameter search (RandomizedSearchCV, n_iter = 10)


In [5]:
pipe = Pipeline([("prep", preprocessor),
                 ("reg", XGBRegressor(objective="reg:squarederror", tree_method="hist",
                                      random_state=RANDOM_STATE, n_jobs=1))])

param_dist = {
    "reg__n_estimators": [200, 300, 400, 600],
    "reg__max_depth": [3, 4, 5, 6, 8],
    "reg__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "reg__subsample": [0.7, 0.8, 0.9, 1.0],
    "reg__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "reg__min_child_weight": [1, 3, 5],
    "reg__gamma": [0, 0.1, 0.3],
    "reg__reg_alpha": [0, 0.1, 1],
    "reg__reg_lambda": [1, 1.5, 2],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, scoring="neg_root_mean_squared_error",
                            cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                            random_state=RANDOM_STATE, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print("Best CV RMSE:", round(-search.best_score_, 2))
print("Best params  :", search.best_params_)


Best CV RMSE: 1110.42
Best params  : {'reg__subsample': 0.7, 'reg__reg_lambda': 2, 'reg__reg_alpha': 1, 'reg__n_estimators': 600, 'reg__min_child_weight': 5, 'reg__max_depth': 4, 'reg__learning_rate': 0.03, 'reg__gamma': 0, 'reg__colsample_bytree': 0.6}


# ## 6. Evaluate on the untouched test set


In [6]:
best = search.best_estimator_
pred = best.predict(X_test)
metrics = {
    "mae": round(mean_absolute_error(y_test, pred), 2),
    "rmse": round(np.sqrt(mean_squared_error(y_test, pred)), 2),
    "r2": round(r2_score(y_test, pred), 4),
    "test_rows": int(len(y_test)),
    "target_mean": round(float(y_test.mean()), 2),
}
print(json.dumps(metrics, indent=2))


{
  "mae": 394.92,
  "rmse": 1013.63,
  "r2": 0.7847,
  "test_rows": 5102,
  "target_mean": 961.82
}


# ## 7. Actual vs predicted & residual analysis


In [7]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, s=6, alpha=0.3)
lim = max(y_test.max(), pred.max())
plt.plot([0, lim], [0, lim], "r--")
plt.xlabel("Actual recovery (USD)"); plt.ylabel("Predicted recovery (USD)")
plt.title(f"Actual vs Predicted (R2 = {metrics['r2']:.3f})")
plt.tight_layout(); plt.savefig(REPORTS / "03_actual_vs_pred.png", dpi=110); plt.show()

residuals = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred, residuals, s=6, alpha=0.3)
axes[0].axhline(0, color="r", linestyle="--")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residual"); axes[0].set_title("Residuals vs predicted")
sns.histplot(residuals, bins=60, ax=axes[1]); axes[1].set_title("Residual distribution")
plt.tight_layout(); plt.savefig(REPORTS / "03_residuals.png", dpi=110); plt.show()


C:\Windows\Temp\ipykernel_31500\2809141402.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "03_actual_vs_pred.png", dpi=110); plt.show()


C:\Windows\Temp\ipykernel_31500\2809141402.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "03_residuals.png", dpi=110); plt.show()


# ## 8. Feature importance


In [8]:
feat_names = best.named_steps["prep"].get_feature_names_out()
imp_df = (pd.DataFrame({"feature": feat_names,
                        "importance": best.named_steps["reg"].feature_importances_})
          .sort_values("importance", ascending=False).reset_index(drop=True))
plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df.head(20), y="feature", x="importance")
plt.title("Recovery model - top 20 features")
plt.tight_layout(); plt.savefig(REPORTS / "03_feature_importance.png", dpi=110); plt.show()
print(imp_df.head(15))


                            feature  importance
0             num__claim_amount_usd    0.203422
1               cat__cpt_code_27447    0.190539
2               cat__cpt_code_99213    0.046671
3               cat__cpt_code_59400    0.044445
4               cat__cpt_code_93454    0.038253
5               cat__cpt_code_29827    0.026726
6               cat__cpt_code_99214    0.025646
7               cat__cpt_code_58150    0.021440
8         cat__auth_status_obtained    0.019003
9     cat__auth_status_not_required    0.018378
10  num__documentation_completeness    0.017692
11     cat__primary_icd10_dx_M17.11    0.015592
12    cat__auth_status_not_obtained    0.015451
13      cat__primary_icd10_dx_C18.9    0.014621
14      cat__primary_icd10_dx_F20.9    0.013774


C:\Windows\Temp\ipykernel_31500\606857723.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "03_feature_importance.png", dpi=110); plt.show()


# ## 9. Save model + metadata


In [9]:
joblib.dump(best, MODELS / "reimbursement_model.pkl")
meta_path = MODELS / "model_metadata.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
meta["recovery_regression"] = {
    "algorithm": "XGBoost Regressor (reg:squarederror)",
    "target": "estimated_recovery_usd",
    "target_note": ("The dataset has no paid/reimbursement amount. This model predicts "
                    "expected RECOVERY for denied claims - the only real monetary label."),
    "best_params": {k: (v if not isinstance(v, (np.integer, np.floating)) else v.item())
                    for k, v in search.best_params_.items()},
    "cv_best_rmse": round(float(-search.best_score_), 2),
    "metrics": metrics,
    "features_used": FEATURES,
    "top_features": imp_df.head(15)["feature"].tolist(),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
}
meta_path.write_text(json.dumps(meta, indent=2))
print("Saved", MODELS / "reimbursement_model.pkl")
print("=== MODEL 2 COMPLETE ===")


Saved D:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\models\reimbursement_model.pkl
=== MODEL 2 COMPLETE ===
